In [1]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torch

model_id = "Qwen/Qwen3-VL-4B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible.")

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=dtype,
    device_map={"": 0},
    attn_implementation="sdpa"
)

processor = AutoProcessor.from_pretrained(model_id)

/mnt/c/Users/darkf/Desktop/Pixtrall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 713/713 [00:11<00:00, 62.98it/s] 


In [3]:
image = "./table_ronde.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": image},
            {"type": "text", "text": "Décris cette image en français."}
        ]
    }
]



In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

inputs.pop("token_type_ids", None)
inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}

streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    streamer=streamer,
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

for token in streamer:
    print(token, end="", flush=True)

thread.join()
print()


Voici une description de l'image en français :

Cette image montre une réunion d'affaires en vue aérienne, avec cinq personnes assises autour d'une table ronde en métal gris. Les participants sont vêtus de manière professionnelle : des costumes, des chemises et des cravates. Sur la table, on observe des documents, des ordinateurs portables, des stylos et des tasses, suggérant une discussion ou une présentation en cours. L'ambiance est sérieuse et concentrée. Le sol est en carrelage gris avec des lignes blanches
